# Constraint satisfaction

**Group E01** — Project 01, Artificial Intelligence 2026-27. Supporting notebook of the main one.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so Python finds the 'src' folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.model import Problem, State, Stop
from src import informed, local_search, csp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

prob = Problem.from_csv("../data_tiny")          # development dataset
prob_small = Problem.from_csv("../data_small")   # larger dataset, used to check that the code scales

## 4.3 Constraint Satisfaction Problem (CSP)

Here the question is different: **which vehicle serves each stop**, without optimising the cost. The visiting order inside each leg is then decided with A* (section 4.1), so the CSP and the search work together.

**Variables.** One per producer visit `(collect, day, producer)` and one per order `(deliver, order)`.

**Domains.** The set of vehicles. *Node consistency* removes the vehicles whose capacity is smaller than the weight of the variable.

**Hard constraints** (a violation makes the assignment invalid):

| Constraint | Type | Where it is checked |
|---|---|---|
| Capacity: stops of one leg and vehicle weigh at most the capacity | resource (Atmost), n-ary | `consistent`, `allowed` |
| Two stops of one leg and vehicle: A before B *or* B before A inside both windows | disjunctive, binary | `compatible` (used by AC-3) |
| The whole leg has a route inside all its windows | global | `leg_ok` (A*) |
| Old deliveries of a vehicle delay its collection | vehicle clock | `clock_ok` |
| Complete plan: flow, warehouse dwell, windows | global, final test | `goal_test` |

**Soft preferences** (they only order the values, they never remove one): perishable products (fruit, vegetables) prefer the refrigerated vehicle; a stop prefers the vehicle that already serves a nearby stop. Which constraints are hard or soft is our interpretation, since the data has no refrigeration rule.

**Search.** Backtracking, day by day (later days depend on earlier ones), with *MRV* and *degree* to choose the variable, *LCV* to order the values, *forward checking* after every assignment and *AC-3* before the search. The number of backtracks is the number of assignments that had to be undone.

In [2]:
model = csp.CSP(prob)
variables = pd.DataFrame([{"variable": v, "weight_kg": round(model.info[v]["weight"], 1),
                           "window_open_h": round(model.info[v]["open"] / 60 % 24, 2),
                           "window_close_h": round(model.info[v]["close"] / 60 % 24, 2),
                           "domain": model.domains[v]} for v in model.vars])
print(len(model.vars), "variables")
variables

18 variables


,variable,weight_kg,window_open_h,window_close_h,domain
0,"(collect, 2026-10-31, 2)",29.9,8.25,10.00,"[1, 2]"
1,"(collect, 2026-10-31, 1)",41.1,9.00,11.25,"[1, 2]"
2,"(collect, 2026-10-31, 3)",32.1,7.75,9.75,"[1, 2]"
3,"(collect, 2026-10-30, 1)",52.0,9.50,9.75,"[1, 2]"
4,"(collect, 2026-10-30, 3)",37.3,8.00,9.75,"[1, 2]"
5,"(collect, 2026-10-30, 2)",77.6,9.00,10.50,"[1, 2]"
6,"(deliver, 1)",22.0,10.50,12.75,"[1, 2]"
7,"(deliver, 2)",6.8,7.25,10.25,"[1, 2]"
8,"(deliver, 3)",21.6,11.50,14.25,"[1, 2]"
9,"(deliver, 4)",26.3,11.50,13.75,"[1, 2]"


In [3]:
def run_csp(problem, variants):
    rows, sols = [], {}
    for name, options in variants.items():
        solution, stats = csp.backtracking_search(csp.CSP(problem), **options)
        feasible = solution is not None and problem.is_feasible_full(solution["state"])
        rows.append({"variant": name, "solved": solution is not None, "feasible": feasible,
                     "cost_km": round(problem.calculate_cost(solution["state"]), 2) if solution else None,
                     "backtracks": stats["backtracks"], "assignments": stats["assignments"],
                     "fc_pruned_values": stats["fc_pruned_values"], "ac3_pruned_values": stats["initial_domain_size"] - stats.get("domain_size_after_ac3", stats["initial_domain_size"]),
                     "seconds": stats["seconds"]})
        sols[name] = solution
    return pd.DataFrame(rows), sols

variants = {"MRV + degree + LCV + FC + AC-3": {},
            "static order + LCV + FC + AC-3": dict(use_mrv=False),
            "MRV + degree, no LCV": dict(use_lcv=False),
            "MRV + degree + LCV, no FC": dict(use_fc=False),
            "MRV + degree + LCV + FC, no AC-3": dict(use_ac3=False),
            "plain backtracking (no heuristics)": dict(use_mrv=False, use_lcv=False, use_fc=False, use_ac3=False)}

csp_tiny, csp_sols_tiny = run_csp(prob, variants)
csp_tiny

,variant,solved,feasible,cost_km,backtracks,assignments,fc_pruned_values,ac3_pruned_values,seconds
0,MRV + degree + LCV + FC + AC-3,True,True,662.47,0,18,1,0,0.006
1,static order + LCV + FC + AC-3,True,True,662.47,0,18,2,0,0.005
2,"MRV + degree, no LCV",True,True,662.47,0,18,1,0,0.005
3,"MRV + degree + LCV, no FC",True,True,662.47,0,18,0,0,0.004
4,"MRV + degree + LCV + FC, no AC-3",True,True,662.47,0,18,1,0,0.005
5,plain backtracking (no heuristics),True,True,662.47,0,18,0,0,0.004


In [4]:
# Same variants on the small dataset
csp_small, csp_sols_small = run_csp(prob_small, variants)
csp_small

,variant,solved,feasible,cost_km,backtracks,assignments,fc_pruned_values,ac3_pruned_values,seconds
0,MRV + degree + LCV + FC + AC-3,True,True,867.5,266,303,129,0,0.144
1,static order + LCV + FC + AC-3,True,True,867.5,0,37,1,0,0.058
2,"MRV + degree, no LCV",True,True,867.5,266,303,129,0,0.142
3,"MRV + degree + LCV, no FC",True,True,867.5,818,855,0,0,0.164
4,"MRV + degree + LCV + FC, no AC-3",True,True,867.5,266,303,129,0,0.142
5,plain backtracking (no heuristics),True,True,867.5,0,37,0,0,0.042


In [5]:
# Assignment found (all heuristics) and the schedule it produces on the tiny dataset
solution = csp_sols_tiny["MRV + degree + LCV + FC + AC-3"]
assignment = pd.DataFrame({"variable": [str(v) for v in solution["assignment"]], "vehicle": list(solution["assignment"].values())})
display(assignment)
prob.route_table(solution["state"]).head(12)

,variable,vehicle
0,"('deliver', 4)",1
1,"('collect', datetime.date(2026, 10, 30), 1)",1
2,"('deliver', 10)",1
3,"('collect', datetime.date(2026, 10, 30), 3)",2
4,"('deliver', 11)",1
5,"('collect', datetime.date(2026, 10, 30), 2)",1
6,"('deliver', 12)",1
7,"('deliver', 1)",1
8,"('deliver', 3)",2
9,"('collect', datetime.date(2026, 10, 31), 2)",1


,vehicle,stop,action,location,arrival,service_start,window,items
0,1,1,pickup,1,30 00:40,30 09:30,30 09:30 - 30 09:45,"[5, 13, 25, 31, 32, 34, 37, 40]"
1,1,2,pickup,2,30 10:11,30 10:11,30 09:00 - 30 10:30,"[12, 23, 27, 28, 30, 33, 36, 38, 39]"
2,1,3,unload,4,30 11:11,30 11:11,,"[5, 13, 25, 31, 32, 34, 37, 40, 12, 23, 27, 28..."
3,2,1,pickup,3,30 00:38,30 08:00,30 08:00 - 30 09:45,"[6, 11, 22, 24, 26, 29, 35]"
4,2,2,unload,4,30 08:48,30 08:48,,"[6, 11, 22, 24, 26, 29, 35]"
5,1,4,load,4,30 11:26,30 11:26,,"[32, 33, 34, 35, 11, 12, 13, 30, 31, 36, 37, 3..."
6,1,5,deliver,5,30 12:06,30 12:06,30 10:15 - 30 14:00,"[32, 33, 34, 35]"
7,1,6,deliver,7,30 12:25,30 12:25,30 11:30 - 30 13:45,"[11, 12, 13]"
8,1,7,deliver,8,30 13:09,30 13:09,30 11:45 - 30 14:00,"[30, 31]"
9,1,8,deliver,6,30 14:07,30 14:07,30 12:15 - 30 14:45,"[36, 37, 38, 39, 40]"


### Observations (executed results)

* **Tiny:** the six variants find a feasible assignment with **0 backtracks** and the same cost, 662.47 km. The problem is loosely constrained, so the heuristics make no visible difference.
* **Small:** plain backtracking and static order need 0 backtracks (37 assignments, one per variable). With MRV + degree the search needs **266 backtracks**, and **818** if forward checking is removed, so forward checking does help (129 values pruned) but MRV did *not* help in this problem. We did not investigate the reason; a hypothesis (not tested) is that MRV does not follow the order of the stops inside a day, so conflicts appear late. LCV makes no difference (266 with and without it).
* **AC-3 pruned 0 values** in both datasets: the binary constraints (capacity of two stops, order of two stops) are too weak to remove a vehicle from a domain. The real conflicts come from the global constraints (`leg_ok`, `clock_ok`), which are checked when a value is assigned and by forward checking.
* Every solution found passes the independent checker (`is_feasible_full`). The cost is not optimised (the soft preferences only order the values): 662.47 km on tiny and 867.50 km on small.
* While developing the CSP we found a constraint that was missing at first: the *old deliveries* of a vehicle delay its collection (`clock_ok`). Without it the search on the small dataset did not finish within 20 000 backtracks.